## Linear Regression — Mini-Batch Gradient Descent

---

### 1. Problem Setup

Assume we have a dataset

$$
\{(\mathbf{x}_1, y_1), (\mathbf{x}_2, y_2), \dots, (\mathbf{x}_N, y_N)\}
$$

where

$$
\mathbf{x}_i \in \mathbb{R}^D
$$

$$
y_i \in \mathbb{R}
$$

The goal is to predict a continuous target $y$ from input $\mathbf{x}$ using linear regression.

---

### 2. Linear Model

The model predicts using a linear combination of weights:

$$
\hat{y}_i = \mathbf{w}^T \mathbf{x}_i
$$

If bias term is included:

$$
\tilde{\mathbf{x}}_i = [1, \mathbf{x}_i]
$$

Then the prediction becomes:

$$
\hat{y}_i = \mathbf{w}^T \tilde{\mathbf{x}}_i
$$

---

### 3. Objective Function (Mean Squared Error)

The loss function for linear regression:

$$
L = \frac{1}{2N} \sum_{i=1}^N (\hat{y}_i - y_i)^2
$$

Or in matrix form:

$$
L = \frac{1}{2N} \| \mathbf{X} \mathbf{w} - \mathbf{y} \|_2^2
$$

---

### 4. Gradient of Loss

The gradient with respect to weights:

$$
\nabla_{\mathbf{w}} L = \frac{1}{N} \mathbf{X}^T (\mathbf{X} \mathbf{w} - \mathbf{y})
$$

---

### 5. Mini-Batch Gradient Descent

- Shuffle the data indices each epoch.
- Split data into mini-batches of size $B$.
- Update weights on each mini-batch:

$$
\mathbf{w} \leftarrow \mathbf{w} - \alpha \, \nabla_{\mathbf{w}} L_\text{batch}
$$

where:

$$
\nabla_{\mathbf{w}} L_\text{batch} = \frac{1}{B} \mathbf{X}_\text{batch}^T (\mathbf{X}_\text{batch} \mathbf{w} - \mathbf{y}_\text{batch})
$$

---

### 6. Weight Initialization

Initialize weights to zeros:

$$
\mathbf{w} = \mathbf{0}
$$

---

### 7. Iterative Optimization

For each epoch:

1. Shuffle the dataset.
2. Generate mini-batches.
3. Compute predictions for the mini-batch:

$$
\hat{\mathbf{y}}_\text{batch} = \mathbf{X}_\text{batch} \mathbf{w}
$$

4. Compute error:

$$
\mathbf{e}_\text{batch} = \hat{\mathbf{y}}_\text{batch} - \mathbf{y}_\text{batch}
$$

5. Compute gradient:

$$
\nabla_{\mathbf{w}} L_\text{batch} = \frac{1}{B} \mathbf{X}_\text{batch}^T \mathbf{e}_\text{batch}
$$

6. Update weights:

$$
\mathbf{w} \leftarrow \mathbf{w} - \alpha \, \nabla_{\mathbf{w}} L_\text{batch}
$$

7. Compute full loss:

$$
L_\text{epoch} = \frac{1}{2N} \| \mathbf{X} \mathbf{w} - \mathbf{y} \|_2^2
$$

---

### 8. Convergence Criterion

Stop early if improvement is smaller than tolerance $\text{tol}$:

$$
|L_\text{current} - L_\text{previous}| < \text{tol}
$$

---

### 9. Prediction

After training, predictions for new data $\mathbf{X}_\text{test}$ are:

$$
\hat{\mathbf{y}}_\text{test} = \mathbf{X}_\text{test} \mathbf{w}
$$

---

### 10. Algorithm Summary

1. Initialize weights: $\mathbf{w} = 0$  
2. Repeat for each epoch:  
    - Shuffle data  
    - Generate mini-batches  
    - Compute batch predictions  
    - Compute gradient  
    - Update weights  
    - Compute full loss  
3. Stop if convergence criterion met  
4. Predict on new data using $\mathbf{w}$

In [1]:
class LinearRegressionMiniBatchGD:
    """
    Linear Regression using Mini-Batch Gradient Descent.

    Parameters
    ----------
    alpha : float, default=0.001
        Learning rate for gradient descent. Must be positive.
    epochs : int, default=1
        Number of passes over the entire training data. Must be a positive integer.
    batch_size : int, default=5
        Size of mini-batches for gradient descent. Must be a positive integer.
    add_bias : bool, default=True
        Whether to include a bias (intercept) term in the model.
    tol : float, default=1e-5
        Tolerance for early stopping. Training stops if loss improvement < tol.
    verbose : bool, default=False
        If True, prints loss after each epoch.

    
    """
    def __init__(self, alpha =0.001 , epochs=1 , batch_size=5, add_bias = True , tol=1e-5, verbose=False):
        self.alpha = alpha
        if self.alpha <=0:
            raise ValueError("alpha needs to be positive")
            
        self.epochs = epochs

        if type(self.epochs) != int or self.epochs <=0:
            raise ValueError("epochs need to be natural number")

        
        self.batch_size = batch_size
        if type(self.batch_size) != int or self.batch_size <=0:
            raise ValueError("batch size needs to be natural number")

        
        self.add_bias = add_bias
        self.tol = tol
        self.verbose = verbose

        # Weights will be initialized during training
        self.weights = None

    def _shuffle(self, N):
        """
        Shuffle indices for mini-batch generation.

        Parameters
        ----------
        N : int
            Number of samples.

        Returns
        -------
        indices : ndarray of shape (N,)
            Shuffled indices.
        """

        indices = np.arange(N)
        np.random.shuffle(indices)

        return indices

    def _generate_mini_batch_indices(self, indices, batch_size):
        """
        Generate indices for mini-batches.

        Parameters
        ----------
        indices : ndarray
            Array of shuffled indices.
        batch_size : int
            Size of each mini-batch.

        Yields
        ------
        batch_indices : ndarray
            Indices of samples for the current mini-batch.
        """
        
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]

            yield batch_indices
        


    def fit(self,X,y):
        """
        Fit the linear regression model using mini-batch gradient descent.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training feature matrix.
        y : ndarray of shape (n_samples,)
            Target values.

        Notes
        -----
        Stops early if loss improvement between epochs is smaller than tol.
        """
        X = np.asarray(X)
        y = np.asarray(y)

        y = y.reshape(-1) # Ensure y is a 1D array

        if X.ndim==1:
            X = X.reshape(-1,1) # Ensure X is 2D


        N , D = X.shape

        # Add bias term if requested

        if self.add_bias:
            X = np.column_stack((np.ones(N),X))
            D +=1

        # Initialize weights
        self.weights = np.zeros(D)
        

        loss_previous = np.inf
        
        for epoch in range(self.epochs):
            # Shuffle indices for this epoch
            shuffled_indices = self._shuffle(N)

            
            # Shuffle indices for this epoch
            for indices in self._generate_mini_batch_indices(shuffled_indices,self.batch_size):
                X_batch , y_batch = X[indices] , y[indices]
                

                y_hat = X_batch @ self.weights
                
                # Compute gradient for mini-batch
                gradient = (X_batch.T @ (y_hat - y_batch)) / len(indices)
                
                # Update weights
                self.weights -= self.alpha * gradient

            # Compute loss over full dataset for monitoring
            loss_current = np.linalg.norm(X @ self.weights - y,ord=2) **2/ (2*N)

            if self.verbose == True:
                print(f'loss for epoch: {epoch} is {loss_current}')

            if abs(loss_current - loss_previous) < self.tol:
                print(f"Stopping early at epoch {epoch}: improvement < {self.tol}")
                print(self.weights)
                break
                
            loss_previous  = loss_current 

    def predict(self,X):
        """
        Predict target values using the trained linear regression model.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Test feature matrix.

        Returns
        -------
        y_pred : ndarray of shape (n_samples,)
            Predicted target values.
        """
        
        if self.weights is None:
            raise ValueError('Model is not fitted yet')
            
        X = np.asarray(X)


        if X.ndim==1:
            X = X.reshape(-1,1) # Ensure X is 2D
            
        N = len(X)
        # Add bias term if model was trained with bias

        if self.add_bias:
            X = np.column_stack((np.ones(N),X))
        # Compute predictions
        return X @ self.weights


## 1. Objective

Aim is to compare **Batch Gradient Descent (BGD)**, **Mini-Batch Gradient Descent (MBGD)** and **Stochastic Gradient Descent (SGD)** in terms of:

- Training time
- Memory usage

Use a synthetic linear regression dataset with **10,000 samples** and **100 features**. Each method is run for **50 repetitions** to compute mean and standard deviation of training time.

Measure **memory consumption** during training.

---

## 2. Method

| Method       | Batch Size | Description |
|--------------|------------|-------------|
| BGD          | N          | Updates weights using **all samples** per iteration. |
| Mini-Batch   | 150        | Updates weights using **batches of 150 samples**. |
| SGD          | 1          | Updates weights using **one sample at a time**. |

Change `batch_size` to switch between methods using the same implementation:

- `batch_size = N` → BGD  
- `batch_size = 1` → SGD  
- `batch_size = 150` → Mini-Batch  

---

## 3. Setup

- Dataset: Synthetic linear regression data  
- Number of epochs: 5  
- Learning rate: 0.01  
- Measurements:
  - Training time (mean and standard deviation)
  - Memory usage (MB)

---

In [2]:
import numpy as np
import pandas as pd
import time
import tracemalloc


np.random.seed(42)
N = 10000  # number of samples
D = 100    # number of features

X = np.random.randn(N, D)
true_weights = np.random.randn(D)
y = X @ true_weights + np.random.randn(N)*0.5  # linear data with noise


num_runs = 50 # run 50 times 

settings = {
    'BGD': N,          # Batch GD: entire dataset
    'MiniBatch': 150,  # Mini-batch GD: batch of 150
    'SGD': 1           # Stochastic GD: batch of 1
}

results = []


for method_name, batch_size in settings.items():
    times = []
    memories = []
    for i in range(num_runs):  
        model = LinearRegressionMiniBatchGD(alpha=0.01, epochs=5, batch_size=batch_size)

        # Track memory
        tracemalloc.start()
        t0 = time.time()
        model.fit(X, y)
        t1 = time.time()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        times.append(t1-t0)
        memories.append(peak / 1024 / 1024)  # convert bytes to MB

    results.append({
        "Method": method_name,
        "Batch Size": batch_size,
        "Mean Time (s)": np.mean(times),
        "Std Time (s)": np.std(times),
        "Memory (MB)": np.mean(memories)
    })

df_results = pd.DataFrame(results)

---

## 4. Results

In [3]:
df_results

,Method,Batch Size,Mean Time (s),Std Time (s),Memory (MB)
0,BGD,10000,0.012167,0.000996,23.425152
1,MiniBatch,150,0.011356,0.000467,8.021097
2,SGD,1,0.537622,0.021481,7.938123


---

## 5. Insights

1. **Training Time**
    - **Mini-Batch and BGD** are extremely fast for this dataset.  
    - **SGD** is much slower because it updates weights per sample.  
    - Mini-Batch can sometimes outperform BGD due to better vectorization of small batches.

2. **Memory Usage**
    - **BGD** consumes the most memory because it uses the entire dataset per update.  
    - **Mini-Batch and SGD** use much less memory as they only load one batch at a time.

3. **Stability**
    - Mini-Batch shows **lowest variability** in training time.  
    - SGD has more fluctuations due to per-sample updates.

---

## 6. Conclusion

- **BGD**: Fast for small datasets, but high memory cost.  
- **Mini-Batch GD**: Best balance between speed, memory, and stability. Recommended for real-world datasets.  
- **SGD**: Very memory-efficient but slower and noisier on small datasets. Suitable for very large or streaming datasets.
---